# Corpus-regression sweep analysis

In [1]:
import polars as pl

from src import get_repo_base
from src.experiments.corpus_regression.analysis import (
    CorpusRegressionAnalysisConfig,
    plot_methods_vs_epoch,
    plot_methods_vs_lookforward,
)
from src.experiments.corpus_regression.config import artifacts_dir

ARTIFACTS = artifacts_dir()

# Print summarize() tables in full (default polars truncation hides columns).
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(64)
pl.Config.set_fmt_str_lengths(80)

polars.config.Config

## Parameters

In [ ]:
# === Parameters ===
NUM_SAMPLES: int = 100_000       # 100_000 or 500_000
GAUSSIAN_STDEV: float = 10000.0      # 1.0 or 0.5
LABEL_TYPE: str = "token_id"   # "rademacher" or "token_id"
NORMALIZE_LABELS: bool = False   # True for [0,1]-normalized token_id labels
LABEL_RANGE: tuple[float, float] = (0.0, 1.0)  # target range when normalized

# Derived output path — namespaced by combo to avoid overwriting
_combo_slug = f"n{NUM_SAMPLES // 1000}k_sigma{GAUSSIAN_STDEV}"
if LABEL_TYPE != "rademacher":
    _combo_slug += f"_{LABEL_TYPE}"
if NORMALIZE_LABELS:
    _combo_slug += "_normalized"
WRITEUP_ASSETS = get_repo_base() / "writeup" / "assets" / "corpus-regression" / _combo_slug

## Supervised-learning

In [ ]:
sl = CorpusRegressionAnalysisConfig.from_sl_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
)
if sl is None:
    print("SL: no artifacts")
else:
    sl.describe("SL")
    print(sl.summarize(metric="corr"))
    print(sl.summarize(metric="mse"))
    display(
        sl.plot_vs_epoch(
            "corr",
            title="SL: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_corr.html",
        )
    )
    display(
        sl.plot_vs_epoch(
            "mse",
            title="SL: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_per_epoch_mse.html",
        )
    )
    display(
        sl.plot_vs_lookforward(
            title="SL: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "sl_vs_lookforward.html",
        )
    )

## GRPO

In [ ]:
grpo = CorpusRegressionAnalysisConfig.from_grpo_sweep(
    artifacts_root=ARTIFACTS, num_samples=NUM_SAMPLES, gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE, normalize_labels=NORMALIZE_LABELS, label_range=LABEL_RANGE,
)
if grpo is None:
    print("GRPO: no artifacts")
else:
    grpo.describe("GRPO")
    print(grpo.summarize(metric="corr"))
    print(grpo.summarize(metric="mse"))
    display(
        grpo.plot_vs_epoch(
            "corr",
            title="GRPO: per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_corr.html",
        )
    )
    display(
        grpo.plot_vs_epoch(
            "mse",
            title="GRPO: per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_per_epoch_mse.html",
        )
    )
    display(
        grpo.plot_vs_lookforward(
            title="GRPO: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "grpo_vs_lookforward.html",
        )
    )

## MaxRL (subtract-baseline + factorized)

In [ ]:
maxrl_sf = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
    artifacts_root=ARTIFACTS,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
)
if maxrl_sf is None:
    print("MaxRL (sub-baseline, factorized): no artifacts")
else:
    maxrl_sf.describe("MaxRL (sub-baseline, factorized)")
    print(maxrl_sf.summarize(metric="corr"))
    print(maxrl_sf.summarize(metric="mse"))
    display(
        maxrl_sf.plot_vs_epoch(
            "corr",
            title="MaxRL (sub-baseline, factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_corr.html",
        )
    )
    display(
        maxrl_sf.plot_vs_epoch(
            "mse",
            title="MaxRL (sub-baseline, factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_per_epoch_mse.html",
        )
    )
    display(
        maxrl_sf.plot_vs_lookforward(
            title="MaxRL (sub-baseline, factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "maxrl_sub_fact_vs_lookforward.html",
        )
    )

### Other MaxRL ablations

Uncomment to inspect the other (`subtract_baseline`, `use_factorized_likelihoods`) combinations if we run these sweeps

In [6]:
# for sub, fact in [(True, False), (False, True), (False, False)]:
#     cfg = CorpusRegressionAnalysisConfig.from_maxrl_sweep(
#         artifacts_root=ARTIFACTS,
#         subtract_baseline=sub,
#         use_factorized_likelihoods=fact,
#         num_samples=NUM_SAMPLES,
#         gaussian_stdev=GAUSSIAN_STDEV,
#     )
#     label = f"MaxRL (sub={sub}, fact={fact})"
#     if cfg is None:
#         print(f"{label}: no artifacts")
#         continue
#     cfg.describe(label)
#     display(cfg.plot_vs_epoch("corr", title=f"{label}: per-epoch (corr)", show_seed_bar=True))
#     display(cfg.plot_vs_lookforward(title=f"{label}: best-epoch vs num_lookforward_tokens", x_scale="uniform", show_seed_bar=True))

## RLOO (factorized)

In [ ]:
rloo_f = CorpusRegressionAnalysisConfig.from_rloo_sweep(
    artifacts_root=ARTIFACTS,
    factorized=True,
    num_samples=NUM_SAMPLES,
    gaussian_stdev=GAUSSIAN_STDEV,
    label_type=LABEL_TYPE,
    normalize_labels=NORMALIZE_LABELS,
    label_range=LABEL_RANGE,
)
if rloo_f is None:
    print("RLOO (factorized): no artifacts")
else:
    rloo_f.describe("RLOO (factorized)")
    print(rloo_f.summarize(metric="corr"))
    print(rloo_f.summarize(metric="mse"))
    display(
        rloo_f.plot_vs_epoch(
            "corr",
            title="RLOO (factorized): per-epoch (corr)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_corr.html",
        )
    )
    display(
        rloo_f.plot_vs_epoch(
            "mse",
            title="RLOO (factorized): per-epoch (mse)",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_per_epoch_mse.html",
        )
    )
    display(
        rloo_f.plot_vs_lookforward(
            title="RLOO (factorized): best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=True,
            save_path=WRITEUP_ASSETS / "rloo_factorized_vs_lookforward.html",
        )
    )

## NTP baseline (intrinsic-variance proxy)

Inference-only: deterministic given the dataset (no seeds, single epoch).
We assemble one config across `candidate_lookforward_tokens` so it slots
into the same plotting helpers as the trained methods.

In [ ]:
from src.data.corpus_regression import candidate_lookforward_tokens

_ntp_per_look = [
    CorpusRegressionAnalysisConfig.from_ntp_baseline(
        artifacts_root=ARTIFACTS,
        num_lookforward_tokens=n,
        num_samples=NUM_SAMPLES,
        label_type=LABEL_TYPE,
        normalize_labels=NORMALIZE_LABELS,
        label_range=LABEL_RANGE,
    )
    for n in candidate_lookforward_tokens
]
_ntp_per_look = [c for c in _ntp_per_look if c is not None]

if not _ntp_per_look:
    ntp = None
    print("NTP baseline: no artifacts")
else:
    ntp = CorpusRegressionAnalysisConfig(
        studies={k: v for c in _ntp_per_look for k, v in c.studies.items()},
        study_seeds={k: v for c in _ntp_per_look for k, v in c.study_seeds.items()},
        study_lookforwards={
            k: v for c in _ntp_per_look for k, v in c.study_lookforwards.items()
        },
    )
    ntp.describe("NTP baseline")
    print(ntp.summarize(metric="corr"))
    print(ntp.summarize(metric="mse"))
    display(
        ntp.plot_vs_lookforward(
            title="NTP baseline: best-epoch vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=False,
            save_path=WRITEUP_ASSETS / "ntp_baseline_vs_lookforward.html",
        )
    )
    display(
        ntp.plot_vs_lookforward(
            metric="mse",
            title="NTP baseline: best-epoch MSE vs num_lookforward_tokens",
            x_scale="uniform",
            show_seed_bar=False,
            save_path=WRITEUP_ASSETS / "ntp_baseline_vs_lookforward_mse.html",
        )
    )

## Cross-method comparison

In [9]:
if any(c is not None for c in (sl, grpo, maxrl_sf, rloo_f)):
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            title="Methods: best-epoch corr vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward.html",
        )
    )
    display(
        plot_methods_vs_lookforward(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            metric="mse",
            title="Methods: best-epoch MSE vs num_lookforward_tokens",
            x_scale="uniform",
            save_path=WRITEUP_ASSETS / "methods_vs_lookforward_mse.html",
        )
    )
else:
    print("No artifacts for any method.")

### Per-epoch training curves (look=1, all methods)

In [10]:
if any(c is not None for c in (sl, grpo, maxrl_sf, rloo_f)):
    display(
        plot_methods_vs_epoch(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            num_lookforward_tokens=1,
            metric="corr",
            show_seed_bar=True,
            title="Methods: per-epoch corr (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_corr.html",
        )
    )
    display(
        plot_methods_vs_epoch(
            sl=sl,
            grpo=grpo,
            maxrl=maxrl_sf,
            rloo=rloo_f,
            num_lookforward_tokens=1,
            metric="mse",
            show_seed_bar=True,
            title="Methods: per-epoch MSE (look=1)",
            save_path=WRITEUP_ASSETS / "methods_vs_epoch_look1_mse.html",
        )
    )
else:
    print("No artifacts for any method.")